[![In Colab öffnen](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Y-Robin/DeepLearningKurs/blob/main/notebooks/Day_6_8/09_tag_6_8_augmentation_regularisierung_lokale_bilder.ipynb)

# Tag 6–8 – 09: Augmentation und Regularisierung mit CIFAR-10

Dieses Beispiel verwendet **8.000 echte CIFAR-10-Trainingsbilder aus zehn Klassen** statt künstlicher Ausschnitte aus zwei Bildern. Die Aufgabe ist deutlich schwieriger: kleine Farbbilder, echte Objekte und visuell ähnliche Klassen wie Katze/Hund oder Auto/Lkw.

Vier CNNs werden unter exakt gleichen Bedingungen verglichen: Baseline, nur Regularisierung, nur Augmentation und die Kombination. Gewählt wird ausschließlich anhand der Validierung; der Testsplit bleibt bis zum Schluss unangetastet. Danach folgen Lernkurven, Confusion Matrix sowie konkrete richtige und falsche Vorhersagen.

## Daten aus dem lokalen Cache und fester Split

F?hre zuerst Notebook 00 aus: Es l?dt beide CIFAR-10-Splits einmalig nach `datasets/`. Dieses Notebook l?dt **nie** Daten aus dem Internet. Fehlt der Cache, erscheint eine klare Hinweis-Meldung. Alle vier Varianten erhalten exakt dieselben Trainings- und Validierungsbilder; nur die jeweils benannte Gegenma?nahme ?ndert sich.

In [ ]:
from pathlib import Path
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import ConfusionMatrixDisplay
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

RANDOM_STATE = 42
torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
plt.rcParams['axes.grid'] = False

DATA_DIR = Path('datasets')
BATCH_SIZE = 64
TRAIN_SAMPLES, VALID_SAMPLES, TEST_SAMPLES = 8_000, 2_000, 2_000
EPOCHS = 12

print('PyTorch:', torch.__version__)
print('Device:', device)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

def loader(dataset, shuffle=False, seed=RANDOM_STATE):
    return DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=0,
                      generator=torch.Generator().manual_seed(seed),
                      pin_memory=torch.cuda.is_available())

def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

# Augmentation ist nur beim Training erlaubt. Validierung und Test bleiben unverändert.
plain_transform = transforms.ToTensor()
augmented_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
])
try:
    plain_data = datasets.CIFAR10(DATA_DIR, train=True, download=False, transform=plain_transform)
    augmented_data = datasets.CIFAR10(DATA_DIR, train=True, download=False, transform=augmented_transform)
    test_data = datasets.CIFAR10(DATA_DIR, train=False, download=False, transform=plain_transform)
except RuntimeError as error:
    raise RuntimeError(
        'CIFAR-10 fehlt in datasets/. Bitte zuerst Notebook 00 ausf?hren; '
        'Beispiel 09 l?dt den Datensatz bewusst nicht selbst herunter.'
    ) from error
CLASS_NAMES = plain_data.classes

indices = torch.randperm(len(plain_data), generator=torch.Generator().manual_seed(RANDOM_STATE))
train_indices = indices[:TRAIN_SAMPLES].tolist()
valid_indices = indices[TRAIN_SAMPLES:TRAIN_SAMPLES + VALID_SAMPLES].tolist()
test_indices = torch.randperm(len(test_data), generator=torch.Generator().manual_seed(RANDOM_STATE + 1))[:TEST_SAMPLES].tolist()
plain_train = Subset(plain_data, train_indices)
augmented_train = Subset(augmented_data, train_indices)
valid_data = Subset(plain_data, valid_indices)
test_subset = Subset(test_data, test_indices)
valid_loader, test_loader = loader(valid_data), loader(test_subset)
print(f'Training: {len(plain_train):,} | Validierung: {len(valid_data):,} | Test: {len(test_subset):,}')
print('Klassen:', ', '.join(CLASS_NAMES))

## Augmentation sichtbar machen

Alle Bilder unten tragen dasselbe Label. Das Modell soll lernen, dass ein Objekt trotz leichter Verschiebung oder Spiegelung zur gleichen Klasse gehört. Diese Transformationen sind für viele natürliche CIFAR-10-Klassen plausibel.

In [ ]:
position = 0
fig, axes = plt.subplots(1, 5, figsize=(12, 2.7))
image, label = plain_train[position]
axes[0].imshow(image.permute(1, 2, 0))
axes[0].set_title(f'Original\n{CLASS_NAMES[label]}')
axes[0].axis('off')
for ax in axes[1:]:
    image, _ = augmented_train[position]
    ax.imshow(image.permute(1, 2, 0))
    ax.set_title('Trainingsvariante')
    ax.axis('off')
plt.tight_layout()

## Vier Varianten: Was hilft der Generalisierung?

Das CNN hat absichtlich rund eine Million Parameter und kann bei 8.000 Beispielen overfitten.

- **Dropout** deaktiviert beim Training zufällig Aktivierungen.
- **Weight Decay (L2)** bremst sehr große Gewichte.
- **Augmentation** erzeugt neue, plausible Ansichten der Trainingsbilder.

Eine regulierte Variante darf eine niedrigere Trainings-Accuracy haben. Entscheidend sind eine bessere Validierungs-Accuracy und eine kleinere Differenz zwischen Training und Validierung.

In [ ]:
class CifarCNN(nn.Module):
    def __init__(self, dropout=0.0):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d((4, 4)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(128 * 4 * 4, 512), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(512, len(CLASS_NAMES)),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

def evaluate(model, data_loader, loss_fn):
    model.eval()
    total_loss = correct = seen = 0
    with torch.no_grad():
        for images, labels in data_loader:
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            total_loss += loss_fn(logits, labels).item() * len(labels)
            correct += (logits.argmax(1) == labels).sum().item()
            seen += len(labels)
    return total_loss / seen, correct / seen

def train_variant(name, train_data, dropout, weight_decay):
    # gleiche Initialisierung: Die Unterschiede kommen nicht vom Zufall beim Start.
    torch.manual_seed(RANDOM_STATE)
    model = CifarCNN(dropout).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=weight_decay)
    loss_fn = nn.CrossEntropyLoss()
    history = []
    for epoch in range(1, EPOCHS + 1):
        model.train()
        loss_sum = correct = seen = 0
        for images, labels in loader(train_data, shuffle=True):
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            logits = model(images)
            loss = loss_fn(logits, labels)
            loss.backward()
            optimizer.step()
            loss_sum += loss.item() * len(labels)
            correct += (logits.argmax(1) == labels).sum().item()
            seen += len(labels)
        valid_loss, valid_acc = evaluate(model, valid_loader, loss_fn)
        row = {'experiment': name, 'epoch': epoch, 'train_acc': correct / seen,
               'valid_acc': valid_acc, 'train_loss': loss_sum / seen, 'valid_loss': valid_loss}
        history.append(row)
        print(f"{name:29s} | Epoche {epoch:02d} | Train {row['train_acc']:.3f} | Valid {valid_acc:.3f}")
    return model, history

experiments = [
    ('Baseline', plain_train, 0.0, 0.0),
    ('Nur Regularisierung', plain_train, 0.45, 1e-3),
    ('Nur Augmentation', augmented_train, 0.0, 0.0),
    ('Augmentation + Regularisierung', augmented_train, 0.45, 1e-3),
]
models, histories = {}, []
started = time.time()
for name, train_data, dropout, weight_decay in experiments:
    print(f'\n{name}: {count_params(CifarCNN(dropout)):,} trainierbare Parameter')
    models[name], history = train_variant(name, train_data, dropout, weight_decay)
    histories.extend(history)
print(f'\nGesamtdauer: {(time.time() - started) / 60:.1f} Minuten')

## Ergebnis vergleichen und Modell auswählen

Die Spalte **Lücke** ist `Train − Valid`. Eine große positive Lücke ist ein Overfitting-Signal. Gewählt wird nur die beste Validierungs-Accuracy – nicht der Testwert.

In [ ]:
history_df = pd.DataFrame(histories)
final = history_df.sort_values('epoch').groupby('experiment').tail(1).copy()
final['gap'] = final['train_acc'] - final['valid_acc']
summary = final[['experiment', 'train_acc', 'valid_acc', 'gap']].sort_values('valid_acc', ascending=False)
display(summary.style.format({'train_acc': '{:.1%}', 'valid_acc': '{:.1%}', 'gap': '{:+.1%}'}))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for name, group in history_df.groupby('experiment', sort=False):
    axes[0].plot(group.epoch, group.train_acc, '--', alpha=.7, label=f'{name} – Train')
    axes[0].plot(group.epoch, group.valid_acc, linewidth=2, label=f'{name} – Valid')
    axes[1].plot(group.epoch, group.valid_loss, linewidth=2, label=name)
axes[0].set(title='Accuracy: Training (gestrichelt) und Validierung', xlabel='Epoche', ylabel='Accuracy', ylim=(0, 1))
axes[1].set(title='Validierungs-Loss', xlabel='Epoche', ylabel='Cross-Entropy-Loss')
axes[0].legend(fontsize=8, ncol=2)
axes[1].legend(fontsize=8)
plt.tight_layout()

best_name = summary.iloc[0].experiment
best_model = models[best_name]
print(f'Gewählt nach Validierung: {best_name}')

## Einmalige Testbewertung und Ergebnisbeispiele

Erst nach der Modellwahl wird der unabhängige Testsplit verwendet. Neben der Test-Accuracy zeigen die Bilder, welche Entscheidungen das Modell trifft. Grün bedeutet richtig, Rot bedeutet falsch. Bei 32×32-Pixel-Bildern sind Fehler wie Katze/Hund oft selbst für Menschen nachvollziehbar.

In [ ]:
def predictions(model, data_loader):
    model.eval()
    all_images, all_true, all_pred = [], [], []
    with torch.no_grad():
        for images, labels in data_loader:
            all_images.append(images)
            all_true.append(labels)
            all_pred.append(model(images.to(device)).argmax(1).cpu())
    return torch.cat(all_images), torch.cat(all_true), torch.cat(all_pred)

test_loss, test_acc = evaluate(best_model, test_loader, nn.CrossEntropyLoss())
images, y_true, y_pred = predictions(best_model, test_loader)
print(f'Test-Accuracy von „{best_name}“: {test_acc:.1%} | Test-Loss: {test_loss:.3f}')

fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay.from_predictions(y_true, y_pred, display_labels=CLASS_NAMES,
    xticks_rotation=45, cmap='Blues', colorbar=False, ax=ax, normalize='true', values_format='.0%')
ax.set_title(f'Confusion Matrix – {best_name} (Zeilen = echte Klasse)')
plt.tight_layout()

def show_examples(indices, title):
    chosen = indices[:8]
    fig, axes = plt.subplots(2, 4, figsize=(11, 5))
    for ax, idx in zip(axes.ravel(), chosen):
        ax.imshow(images[idx].permute(1, 2, 0))
        correct = y_true[idx] == y_pred[idx]
        colour = 'forestgreen' if correct else 'crimson'
        ax.set_title(f'echt: {CLASS_NAMES[y_true[idx]]}\nModell: {CLASS_NAMES[y_pred[idx]]}', color=colour, fontsize=9)
        ax.axis('off')
    for ax in axes.ravel()[len(chosen):]:
        ax.axis('off')
    fig.suptitle(title, fontsize=14)
    plt.tight_layout()

show_examples(torch.where(y_true == y_pred)[0].tolist(), 'Ergebnisbeispiele: richtig klassifiziert')
show_examples(torch.where(y_true != y_pred)[0].tolist(), 'Ergebnisbeispiele: Fehlklassifikationen')

## Fazit

- Beurteile Regularisierung und Augmentation über die **Validierung**, nicht über die Trainings-Accuracy.
- Hilft Regularisierung, wird die Generalisierungslücke typischerweise kleiner.
- Hilft Augmentation, kann Training zunächst schwerer werden, während die Validierung robuster wird.
- Die Kombination muss nicht immer gewinnen: Bei schlechteren Werten sind Dropout, Weight Decay oder die Transformationen möglicherweise zu stark. Ändere dann jeweils nur **eine** Einstellung.

Für einen schnellen technischen Test kann `EPOCHS = 4` gesetzt werden; für die eigentliche Diskussion anschließend wieder mindestens 12 Epochen verwenden.